# Welcome to the Jupyter Notebook for Ro-Vibrational Spectroscopy!
### Created Fall 2022: J. A. DePaolo-Boisvert
### Updated Spring 2024

This notebook guides you through calculating and simulating the infrared absorption spectrum of HCl
using quantum mechanical models for molecular vibration and rotation.  
You will retrieve real spectroscopic constants from NIST and compare your calculated spectrum to
experimental data.

In [ ]:
# Importing packages — collections of pre-written code useful for scientific computing
import numpy as np
import scipy as sp
import scipy.constants as cons  # constants is a submodule of scipy
import matplotlib as mp
import matplotlib.pyplot as plt
import os
%matplotlib inline

def plot_data(title, xs, ys, xlabel, ylabel):
    plt.clf()
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    _ = plt.plot(xs, ys)
    plt.show()

## Unit Reference

Throughout this notebook the following internal units and conversions are used:

| Quantity | Internal Unit | Display Unit | Conversion Factor |
|---|---|---|---|
| Mass | kg | amu | `amutokg = 1.66e-27 kg/amu` |
| Vibrational frequency | Hz | cm\(^{-1}\) | divide by `c * 100` |
| Energy | J | cm\(^{-1}\) | divide by `h * c * 100`  (= `cmm1toJ`) |
| Bond length | m | \(\unicode{x212B}\) (Angstrom) | multiply by `1e10` |

The NIST spectroscopic constants (\(\nu_e, B_e, D_e, \alpha_e\)) are all given in **cm\(^{-1}\)**.  
The dimensionless anharmonicity constant \(\chi_e\) is unitless (it appears as the product \(\nu_e \chi_e\) in the energy equation).

In [ ]:
# ── Fundamental Constants ──────────────────────────────────────────────────
h   = cons.Planck      # J·s
kb  = cons.Boltzmann   # J/K
c   = cons.c           # m/s

# ── Unit Conversions ──────────────────────────────────────────────────────
amutokg  = 1.66e-27          # kg / amu
evtojoule = 1.602176634e-19  # J / eV
cmm1toJ  = h * c * 100       # J per cm^-1  (used for cm^-1 <-> J conversions)

# ── Diatomic Molecule: H-35Cl ─────────────────────────────────────────────
# Atomic masses in amu, converted to kg
mass1      = 1.007825  * amutokg   # H
mass2      = 34.968853 * amutokg   # 35Cl
mass2_37   = 36.965903 * amutokg   # 37Cl  (for isotopologue comparison)
# Other common masses: D=2.014102 amu

# ── Bond Force Constant & Temperature ────────────────────────────────────
k_force = 481   # N/m  (approximate for HCl)
temp    = 300   # K

# ── Reduced Masses ────────────────────────────────────────────────────────
reduced_mass = lambda m1, m2: (m1 * m2) / (m1 + m2)

mu     = reduced_mass(mass1, mass2)     # H-35Cl
mu_37  = reduced_mass(mass1, mass2_37)  # H-37Cl

print(f"Reduced mass H-35Cl : {mu:.6e} kg")
print(f"Reduced mass H-37Cl : {mu_37:.6e} kg")

### Harmonic Vibrational Frequency

From classical mechanics, the vibrational frequency of a harmonic oscillator is:

$$f = \frac{1}{2\pi}\sqrt{\frac{k}{\mu}}$$

where \(k\) is the force constant and \(\mu\) is the **reduced mass** of the two atoms.

In [ ]:
# Harmonic frequency function
Freq = lambda k, mu: np.sqrt(k / mu) / (2 * cons.pi)  # returns Hz

# H-35Cl harmonic frequency
freq_vib      = Freq(k_force, mu)                     # Hz
freq_vib_wvnm = freq_vib / (c * 100)                  # convert Hz -> cm^-1

# H-37Cl harmonic frequency
freq_vib_37      = Freq(k_force, mu_37)               # Hz
freq_vib_wvnm_37 = freq_vib_37 / (c * 100)            # convert Hz -> cm^-1

print(f"H-35Cl harmonic frequency: {freq_vib:.4e} Hz  |  {freq_vib_wvnm:.2f} cm^-1")
print(f"H-37Cl harmonic frequency: {freq_vib_37:.4e} Hz  |  {freq_vib_wvnm_37:.2f} cm^-1")
print(f"Isotope frequency shift (35Cl - 37Cl): {freq_vib_wvnm - freq_vib_wvnm_37:.2f} cm^-1")

### Reflection Questions — Harmonic Frequency

1. **Why do we use the reduced mass \(\mu\) rather than the mass of either individual atom?**  
   *Your Answer:*

2. **Why will this harmonic frequency not exactly match the \(\nu_e\) value you find in the NIST table?**  
   *Your Answer:*

3. **Does H-37Cl vibrate at a higher or lower frequency than H-35Cl? Explain in terms of mass.**  
   *Your Answer:*

### Harmonic Oscillator Energy Levels (Equation 1)

The energy levels of a quantum harmonic oscillator are:

$$E(\nu_0, \nu) = h\nu_0\left(\nu + \frac{1}{2}\right)$$

The \(+\frac{1}{2}\) means even the lowest level (\(\nu=0\)) has non-zero **zero-point energy** — a purely quantum mechanical effect.

In [ ]:
e_vib_levels = lambda base_freq, level: h * base_freq * (level + 0.5)  # returns J

# First 10 vibrational energy levels of H-35Cl
E_vib = e_vib_levels(freq_vib, np.arange(0, 10))  # J

print("Vibrational level energies (cm^-1):")
for v, E in enumerate(E_vib):
    print(f"  v={v}: {E / cmm1toJ:.2f} cm^-1")  # convert J -> cm^-1

### Rigid Rotor Energy Levels (Equation 2)

$$E(B_e, J) = \frac{h^2}{8\pi^2 I}J(J+1) = B_e J(J+1)$$

where \(I = \mu r^2\) is the moment of inertia and \(B_e\) is the rotational constant.  

> **Note:** `r_nuc` and `B_e` (from NIST) encode the **same physical information** — the equilibrium bond length. Once you have \(B_e\) from NIST, you do not need to use the hardcoded bond length again. The NIST value is more precise because it is derived from the actual experimental spectrum.

In [ ]:
# Approximate bond length (used only here for the rigid-rotor illustration)
r_nuc = 1.27e-10  # m  (1.27 Angstroms)

# Moment of inertia
Moment = lambda r, mu: mu * (r ** 2)  # kg·m^2
mom_I  = Moment(r_nuc, mu)

# Rotational constant Be from bond length (J)
fB_e = lambda r: (h ** 2) / (8 * (cons.pi ** 2) * mu * (r ** 2))  # J

# Rotational energy levels
e_rot_levels = lambda B_e_J, level: B_e_J * level * (level + 1)  # J

E_rot = e_rot_levels(fB_e(r_nuc), np.arange(0, 10))  # J

print("Rotational level energies (cm^-1):")
for j, E in enumerate(E_rot):
    print(f"  J={j}: {E / cmm1toJ:.4f} cm^-1")  # convert J -> cm^-1

## Energy Level Diagram (Harmonic / Rigid-Rotor Approximation)

The diagram below is generated from the **classical harmonic oscillator** and **rigid rotor** approximations
using only the force constant and bond length defined above — no NIST parameters are needed.
It illustrates the structure of R-branch (\(\Delta J = +1\)) and P-branch (\(\Delta J = -1\)) transitions.

In [ ]:
# ── Energy Level Diagram ───────────────────────────────────────────────────
# Uses harmonic/rigid-rotor approximations (no NIST parameters required)

B_e_approx = fB_e(r_nuc) / cmm1toJ  # convert J -> cm^-1 for display

n_vib_show = 6   # vibrational levels to draw
n_rot_show = 7   # rotational sublevels per vibrational state

# Vibrational energies (cm^-1)
vib_energies = np.array([freq_vib_wvnm * (v + 0.5) for v in range(n_vib_show)])
# Rotational energies (cm^-1) relative to vib level
rot_offsets  = np.array([B_e_approx * J * (J + 1) for J in range(n_rot_show)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 7))
fig.suptitle("H-35Cl Energy Level Diagram (Harmonic/Rigid-Rotor Approximation)", fontsize=13)

# ── Left panel: vibrational levels ──────────────────────────────────────
for v, E in enumerate(vib_energies):
    ax1.hlines(E, 0.1, 0.9, colors='black', linewidth=2)
    ax1.text(0.92, E, f"v={v}", va='center', fontsize=9)

# Zero-point energy annotation
ax1.annotate(
    f"ZPE = {vib_energies[0]:.0f} cm\u207b\u00b9",
    xy=(0.5, vib_energies[0]), xytext=(0.05, vib_energies[0] - 200),
    arrowprops=dict(arrowstyle='->', color='blue'), color='blue', fontsize=9
)

# Spacing label
ax1.annotate(
    "", xy=(0.15, vib_energies[1]), xytext=(0.15, vib_energies[0]),
    arrowprops=dict(arrowstyle='<->', color='gray')
)
ax1.text(0.17, (vib_energies[0] + vib_energies[1]) / 2,
         f"h\u03bd\u2080 = {freq_vib_wvnm:.0f} cm\u207b\u00b9", va='center', fontsize=8, color='gray')

ax1.set_xlim(0, 1.2)
ax1.set_ylabel("Energy (cm\u207b\u00b9)")
ax1.set_title("Vibrational Levels")
ax1.set_xticks([])

# ── Right panel: rotational sublevels within v=0 (blue) and v=1 (red) ───
colors_vib = ['blue', 'red']
x_left  = [0.05, 0.55]   # x-start of horizontal lines per vibrational state
x_right = [0.45, 0.95]
x_mid   = [0.25, 0.75]

for vi in range(2):
    E_vib_ref = vib_energies[vi]
    col = colors_vib[vi]
    for J in range(n_rot_show):
        E_tot = E_vib_ref + rot_offsets[J]
        ax2.hlines(E_tot, x_left[vi], x_right[vi], colors=col, linewidth=1.5)
        ax2.text(x_right[vi] + 0.01, E_tot, f"J={J}", va='center', fontsize=7, color=col)

# Draw example R-branch arrow: v=0,J=2 -> v=1,J=3  (Delta J = +1)
E_r_lower = vib_energies[0] + rot_offsets[2]
E_r_upper = vib_energies[1] + rot_offsets[3]
ax2.annotate(
    "", xy=(0.60, E_r_upper), xytext=(0.40, E_r_lower),
    arrowprops=dict(arrowstyle='->', color='green', lw=2)
)
ax2.text(0.48, (E_r_lower + E_r_upper) / 2, "R\n(\u0394J=+1)", fontsize=8, color='green', ha='center')

# Draw example P-branch arrow: v=0,J=3 -> v=1,J=2  (Delta J = -1)
E_p_lower = vib_energies[0] + rot_offsets[3]
E_p_upper = vib_energies[1] + rot_offsets[2]
ax2.annotate(
    "", xy=(0.58, E_p_upper), xytext=(0.42, E_p_lower),
    arrowprops=dict(arrowstyle='->', color='orange', lw=2)
)
ax2.text(0.38, (E_p_lower + E_p_upper) / 2, "P\n(\u0394J=\u22121)", fontsize=8, color='orange', ha='center')

ax2.set_xlim(0, 1.1)
ax2.set_title("Rotational Sublevels (v=0 blue, v=1 red)")
ax2.set_ylabel("Energy (cm\u207b\u00b9)")
ax2.set_xticks([])

plt.tight_layout()
plt.show()

### Reflection Questions — Energy Level Diagram

4. **Compare the spacing of vibrational levels vs. rotational levels. Which is larger, and by roughly what factor?**  
   *Your Answer:*

5. **Why are there many rotational levels (different \(J\) values) within a single vibrational state?**  
   *Your Answer:*

6. **The selection rule \(\Delta J = \pm 1\) means there is no Q-branch (\(\Delta J = 0\)) in HCl. Why is \(\Delta J = 0\) forbidden for a diatomic molecule with a permanent dipole moment vibrating along its bond axis?**  
   *Your Answer:*

## Full Rovibrational Energy Levels — \(T(\nu, J)\)

### From Equation 4 of the text

Adding anharmonicity, centrifugal distortion, and vibration-rotation coupling corrections gives:

$$T(\nu,J) = \nu_e\left(\nu+\frac{1}{2}\right) + B_e J(J+1) - \nu_e\chi_e\left(\nu+\frac{1}{2}\right)^2 - D_e J^2(J+1)^2 - \alpha_e\left(\nu+\frac{1}{2}\right)J(J+1)$$

All five parameters are in **cm\(^{-1}\)**, so \(T\) is also in cm\(^{-1}\).

| Term | Formula | Physical Meaning |
|---|---|---|
| Harmonic vibration | \(\nu_e(\nu+\frac{1}{2})\) | Energy of an ideal spring; equal spacing between levels |
| Rigid rotation | \(B_e J(J+1)\) | Kinetic energy of rotation assuming fixed bond length |
| Anharmonicity | \(-\nu_e\chi_e(\nu+\frac{1}{2})^2\) | Real bonds are not perfect springs; levels get closer together at high \(\nu\) |
| Centrifugal distortion | \(-D_e J^2(J+1)^2\) | Fast rotation stretches the bond, increasing \(I\) and reducing \(B\); \(D_e \approx 5\times10^{-4}\) cm\(^{-1}\) for HCl |
| Vibration-rotation coupling | \(-\alpha_e(\nu+\frac{1}{2})J(J+1)\) | Higher vibrational states have longer average bond length, reducing \(B_e\) |

In [ ]:
# T(nu, J) in cm^-1  — all parameters must be in cm^-1
T_levels = lambda nu, J, nu_e, chi_e, B_e, D_e, a_e: (
    nu_e * (nu + 0.5)                     # harmonic vibration
    + B_e * J * (J + 1)                   # rigid rotation
    - nu_e * chi_e * (nu + 0.5) ** 2      # anharmonicity correction
    - D_e * J ** 2 * (J + 1) ** 2         # centrifugal distortion
    - a_e * (nu + 0.5) * J * (J + 1)      # vibration-rotation coupling
)

## NIST Lookup Instructions

Follow these steps to find the five spectroscopic constants for H-35Cl:

1. Go to the [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) and search for **"HCl"**.
2. Click the entry for hydrogen chloride and navigate to **"Constants of Diatomic Molecules"**.
3. Find the row for **H-35Cl** (the most abundant isotopologue).
4. Record the five constants listed below. All are in **cm\(^{-1}\)** except \(\chi_e\) which is dimensionless.

| Symbol | NIST Column Header | Physical Meaning | Typical Magnitude for HCl |
|---|---|---|---|
| \(\nu_e\) | \(\omega_e\) | Harmonic vibrational frequency | ~2990 cm\(^{-1}\) |
| \(\chi_e\) | \(\omega_e x_e / \omega_e\) | Anharmonicity (dimensionless) | ~0.017 |
| \(B_e\) | \(B_e\) | Rotational constant | ~10.6 cm\(^{-1}\) |
| \(D_e\) | \(D_e\) | Centrifugal distortion constant | ~**5\(\times\)10\(^{-4}\)** cm\(^{-1}\) (very small!) |
| \(\alpha_e\) | \(\alpha_e\) | Vibration-rotation coupling constant | ~0.3 cm\(^{-1}\) |

> **Important:** \(D_e\) is a very small number, approximately \(5 \times 10^{-4}\) cm\(^{-1}\).  
> Do not confuse it with the dissociation energy \(D_0\) or \(D_e\) (equilibrium dissociation energy), which are much larger (~4 eV).  
> Enter \(D_e\) exactly as shown in the table (e.g., `5.32e-4`).

In [ ]:
# ── NIST Spectroscopic Constants for H-35Cl ───────────────────────────────
# Fill in each value from the NIST Constants of Diatomic Molecules table.
# ALL values in cm^-1  (chi_e is dimensionless)

nu_e  = 2990.946   # harmonic vibrational frequency  (cm^-1)
chi_e = 52.8186 / 2990.946   # anharmonicity constant           (dimensionless)
B_e   = 10.59341   # rotational constant              (cm^-1)
D_e   = 5.3194e-4   # centrifugal distortion constant  (cm^-1, ~5e-4 for HCl)
a_e   = 0.30718   # vibration-rotation coupling      (cm^-1)

# ── Guard: this will raise a clear error if any value is still None ───────
assert all(v is not None for v in [nu_e, chi_e, B_e, D_e, a_e]), \
    "Fill in all five NIST spectroscopic constants before continuing!"

In [ ]:
# ── Compute T(nu, J) for H-35Cl ───────────────────────────────────────────
# Number of levels
nu_max = 2   # vibrational levels (0 and 1)
J_max  = 25  # rotational levels (J = 0..24)

# levels[nu, J] in cm^-1
levels = np.array([
    [T_levels(i, j, nu_e, chi_e, B_e, D_e, a_e) for j in range(J_max)]
    for i in range(nu_max)
])

print("Energy level matrix shape (nu, J):", levels.shape)
print(levels)

## Boltzmann Distribution

At thermal equilibrium, the probability of a molecule being in energy level \(i\) is:

$$P_i = e^{\frac{-E_i}{k_B T}}$$

Higher energy levels are exponentially less populated at room temperature.

In [ ]:
boltzmann_prob = lambda energies_J, temp: np.exp(-energies_J / (cons.Boltzmann * temp))

### Boltzmann Degeneracy

The rigid rotor wave functions are spherical harmonics — each \(J\) level has \(2J+1\) degenerate
magnetic sub-states (\(m_J = -J, \ldots, +J\)), just like electron orbital degeneracy.  
The effective population of level \(J\) is therefore:

$$N_J \propto (2J+1)\, e^{-E_J / k_B T}$$

In [ ]:
rot_degeneracy = lambda J: 2 * J + 1

# Convert cm^-1 -> J before passing to Boltzmann
boltzmann_pop = (
    boltzmann_prob(levels * cmm1toJ, temp)  # convert cm^-1 -> J
    * rot_degeneracy(np.arange(levels.shape[1]))
)

# Normalize so all populations sum to 1
normalized_boltzmann_pop = boltzmann_pop / np.sum(boltzmann_pop)
print(normalized_boltzmann_pop)

print("Sum of normalized populations:", np.sum(normalized_boltzmann_pop))
print("Vibrational state populations (sum over J):", np.sum(normalized_boltzmann_pop, axis=1))
print("Top 5 rotational state populations (sum over nu):")
rot_pops = np.sum(normalized_boltzmann_pop, axis=0)
top5 = np.argsort(rot_pops)[::-1][:5]
for j in top5:
    print(f"  J={j}: {rot_pops[j]:.5f}")

In [ ]:
# Plot the Boltzmann rotational population distribution
plt.clf()
plt.bar(np.arange(J_max), np.sum(normalized_boltzmann_pop, axis=0))
plt.xlabel("Rotational Quantum Number J")
plt.ylabel("Normalized Population")
plt.title("Boltzmann Rotational Population Distribution — H-35Cl at 300 K")
plt.show()


### Reflection Questions — Boltzmann Population

7. **\(J=0\) does not have the highest population even though it is the lowest energy level. Explain why.**  
   *Your Answer:*

8. **How does the Boltzmann population plot predict which spectral peaks will be tallest in the simulated spectrum?**  
   *Your Answer:*

9. **How would you check whether `J_max = 25` is large enough to capture essentially all of the population? What would you change if it were not?**  
   *Your Answer:*

## Selection Rules for Spectroscopy

A vibrational transition is IR-active only if it changes the dipole moment.  
For the ro-vibrational spectrum the simultaneous selection rules are:

- \(\Delta\nu = +1\) (absorption from ground vibrational state)
- \(\Delta J = +1\) → **R-branch** (higher energy side of spectrum)
- \(\Delta J = -1\) → **P-branch** (lower energy side of spectrum)
- \(\Delta J = 0\) is **forbidden** for diatomic molecules with no electronic angular momentum → no **Q-branch**

In [ ]:
# Transition energies in cm^-1
# R-branch: v=0,J -> v=1,J+1  (Delta J = +1)
r_transitions = levels[1, 1:] - levels[0, :-1]   # cm^-1
# P-branch: v=0,J -> v=1,J-1  (Delta J = -1)
p_transitions = levels[1, :-1] - levels[0, 1:]   # cm^-1

# Boltzmann weights for each transition's lower level
r_weights = normalized_boltzmann_pop[0, :-1]  # lower state is v=0, J=0..J_max-2
p_weights = normalized_boltzmann_pop[0, 1:]   # lower state is v=0, J=1..J_max-1

print("R-branch transition energies (first 5, cm^-1):", r_transitions[:5])
print("P-branch transition energies (first 5, cm^-1):", p_transitions[:5])

In [ ]:
plt.clf()
plt.scatter(np.arange(J_max), levels[1, :] - levels[0, :], label='Q-branch (Pure vib, forbidden)')
plt.scatter(np.arange(1, J_max), p_transitions, label='P-branch (\u0394J=-1)')
plt.scatter(np.arange(J_max - 1), r_transitions, label='R-branch (\u0394J=+1)')
plt.legend()
plt.xlabel('Base J level')
plt.ylabel('Transition Energy (cm\u207b\u00b9)')
plt.title('HCl Ro-Vibrational Transition Energies')
plt.show()

## Generating a Simulated Spectrum

Each transition is modeled as a Gaussian peak:

$$f(\lambda) = W_b \, e^{-\frac{(\lambda - \lambda_0)^2}{2\sigma^2}}$$

where \(W_b\) is the Boltzmann weight of the lower state and \(\sigma\) is an empirical linewidth.

### Linewidth Parameter \(\sigma\)

The parameter `sigma = 2.5` cm\(^{-1}\) is **empirical** — chosen to match the observed peak widths in the
NIST spectrum. In a real gas-phase experiment, line broadening arises from three physical sources:

- **Doppler broadening**: molecules moving toward/away from the light source see a shifted frequency
  (Gaussian profile, dominant at low pressure).
- **Pressure (collisional) broadening**: collisions interrupt emission/absorption and shorten the
  effective coherence time via the uncertainty principle (Lorentzian profile, grows with pressure).
- **Natural linewidth**: the excited state has a finite lifetime, giving an intrinsic Lorentzian
  broadening (usually negligible in the IR).

The convolution of Gaussian and Lorentzian profiles is a **Voigt profile**, but a simple Gaussian
is a reasonable approximation at the resolution of typical student-grade FTIR instruments.

In [ ]:
# Gaussian peak function: a=amplitude, b=center, c=width (sigma)
Gaussian = lambda x, a, b, c: a * np.exp(-1 * (x - b) ** 2 / (2 * c ** 2))

In [ ]:
# Spectral domain and linewidth
domain = np.linspace(2500, 3200, 2000)  # cm^-1, 2000 points for smoother curve
sigma  = 0.5                             # cm^-1  (empirical linewidth)

# Build H-35Cl spectrum as a sum of Gaussians
spectrum_35 = np.zeros_like(domain)

for transition, weight in zip(r_transitions, r_weights):
    spectrum_35 += Gaussian(domain, weight, transition, sigma)

for transition, weight in zip(p_transitions, p_weights):
    spectrum_35 += Gaussian(domain, weight, transition, sigma)

# Plot absorbance and transmittance
plot_data('H-35Cl Simulated Absorbance Spectrum', domain, spectrum_35,
          'Wavenumber (cm\u207b\u00b9)', 'Absorbance (arb. units)')
plot_data('H-35Cl Simulated Transmittance Spectrum', domain, 1 - spectrum_35,
          'Wavenumber (cm\u207b\u00b9)', 'Transmittance')

## Isotope Effects: H-37Cl

Natural chlorine is approximately **75.77% \(^{35}\)Cl** and **24.23% \(^{37}\)Cl**.  
Because the two isotopologues have slightly different reduced masses, they have slightly different
vibrational and rotational constants — producing a characteristic **doublet** structure that is
visible in the NIST spectrum of HCl.

The two isotopologue peaks are separated by only a few cm\(^{-1}\), so a spectrometer with
sufficient resolution can resolve the doublet.  
In the section below you can optionally fill in the NIST constants for H-37Cl
to add the second isotopologue to the simulated spectrum.

In [ ]:
# ── Optional: NIST constants for H-37Cl ──────────────────────────────────
# Leave as None to skip; the natural-abundance mixture plot will use only H-35Cl.

nu_e_37  = None   # cm^-1
chi_e_37 = None   # dimensionless
B_e_37   = None   # cm^-1
D_e_37   = None   # cm^-1
a_e_37   = None   # cm^-1

HAS_37CL = all(v is not None for v in [nu_e_37, chi_e_37, B_e_37, D_e_37, a_e_37])
print("H-37Cl constants provided:", HAS_37CL)

In [ ]:
# ── H-37Cl spectrum (computed only if constants were filled in) ───────────
ABUNDANCE_35 = 0.7577  # natural abundance of H-35Cl
ABUNDANCE_37 = 0.2423  # natural abundance of H-37Cl

if HAS_37CL:
    levels_37 = np.array([
        [T_levels(i, j, nu_e_37, chi_e_37, B_e_37, D_e_37, a_e_37) for j in range(J_max)]
        for i in range(nu_max)
    ])

    r_transitions_37 = levels_37[1, 1:] - levels_37[0, :-1]
    p_transitions_37 = levels_37[1, :-1] - levels_37[0, 1:]

    # Boltzmann populations for H-37Cl
    bp_37 = (
        boltzmann_prob(levels_37 * cmm1toJ, temp)  # convert cm^-1 -> J
        * rot_degeneracy(np.arange(J_max))
    )
    nbp_37 = bp_37 / np.sum(bp_37)

    r_weights_37 = nbp_37[0, :-1]
    p_weights_37 = nbp_37[0, 1:]

    spectrum_37 = np.zeros_like(domain)
    for transition, weight in zip(r_transitions_37, r_weights_37):
        spectrum_37 += Gaussian(domain, weight, transition, sigma)
    for transition, weight in zip(p_transitions_37, p_weights_37):
        spectrum_37 += Gaussian(domain, weight, transition, sigma)

    # Natural-abundance mixture
    spectrum_mix = ABUNDANCE_35 * spectrum_35 + ABUNDANCE_37 * spectrum_37

    # ── Plot all three on the same axes ───────────────────────────────────
    print(domain)
    plt.clf()
    plt.plot(domain, spectrum_35,  label='H-35Cl only',           alpha=0.7)
    plt.plot(domain, spectrum_37,  label='H-37Cl only',           alpha=0.7)
    plt.plot(domain, spectrum_mix, label='Natural abundance mix', linewidth=2)
    plt.xlabel('Wavenumber (cm\u207b\u00b9)')
    plt.ylabel('Absorbance (arb. units)')
    plt.title('HCl Isotopologue Comparison')
    plt.legend()
    plt.show()

else:
    print("H-37Cl constants not filled in — plotting H-35Cl only.")
    plot_data('H-35Cl Simulated Absorbance Spectrum', domain, spectrum_35,
              'Wavenumber (cm\u207b\u00b9)', 'Absorbance (arb. units)')

In [ ]:
import matplotlib.pyplot as plt
# Plot 1: Natural abundance mix only (green)
plt.clf()
plt.plot(domain, spectrum_mix, color='green', linewidth=2)
plt.xlabel('Wavenumber (cm?¹)')
plt.ylabel('Absorbance (arb. units)')
plt.title('Natural Abundance Mix')
plt.xlim(2500, 3200)
plt.show()

# Natural abundance mix - Zoomed 2900-3000
plt.clf()
plt.plot(domain, spectrum_mix, color='green', linewidth=2)
plt.xlabel('Wavenumber (cm?¹)')
plt.ylabel('Absorbance (arb. units)')
plt.title('Natural Abundance Mix (Zoomed: 2900–3000 cm?¹)')
plt.xlim(2900, 3000)
plt.show()

# Plot 2: H-35Cl and H-37Cl together
plt.clf()
plt.plot(domain, spectrum_35, label='H-35Cl only', alpha=0.7)
plt.plot(domain, spectrum_37, label='H-37Cl only', alpha=0.7)
plt.xlabel('Wavenumber (cm?¹)')
plt.ylabel('Absorbance (arb. units)')
plt.title('H-35Cl vs H-37Cl')
plt.legend()
plt.xlim(2500, 3200)
plt.show()


### Reflection Questions — Isotope Effects

10. **Why do H-35Cl and H-37Cl absorb at slightly different frequencies, even though only the chlorine isotope changes?**  
    *Your Answer:*

11. **What spectral resolution (in cm\(^{-1}\)) would a spectrometer need in order to resolve the isotopologue doublet? How does this compare to the 2.5 cm\(^{-1}\) sigma used in the simulation?**  
    *Your Answer:*

## Loading Real NIST Data

Download the JCAMP-DX file for HCl from NIST:  
[https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=C7647010&Index=0&Type=IR](https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=C7647010&Index=0&Type=IR)

Save the file as `7647-01-0-IR.jdx` in the **same folder as this notebook**, or mount your Google Drive
and update `hcl_jcamp` with the correct path.

In [ ]:
# ── File Loading (works in Colab and locally) ─────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    hcl_jcamp = '/content/drive/MyDrive/7647-01-0-IR.jdx'
except ImportError:
    # Not running in Colab — use a local path (same directory as this notebook)
    hcl_jcamp = os.path.join(os.path.dirname(os.path.abspath('__file__')), '7647-01-0-IR.jdx')

if not os.path.exists(hcl_jcamp):
    raise FileNotFoundError(
        f"JCAMP file not found at: {hcl_jcamp}\n"
        "Download it from: https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=C7647010&Index=0&Type=IR\n"
        "Then save it as '7647-01-0-IR.jdx' in the same folder as this notebook."
    )

print("JCAMP file found:", hcl_jcamp)

In [ ]:
def get_jcamp_data(jcamp_fn):
    """Parse a JCAMP-DX IR file and return (x_data, y_data) as numpy arrays.

    The JCAMP X++(Y..Y) format stores one x-value followed by a variable
    number of y-values on each line.  We average the y-values on each line
    to obtain a single y per x point.
    """
    with open(jcamp_fn, 'r') as f:
        lines = [line.rstrip('\n') for line in f.readlines()]

    data_start = lines.index("##XYDATA=(X++(Y..Y))") + 1
    data_end   = lines.index("##END=")
    data_lines = lines[data_start:data_end]

    # JCAMP rows have variable column counts — use list comprehensions safe for ragged rows
    x_data = np.array([float(line.split()[0]) for line in data_lines])         # first column = x
    y_data = np.array([np.mean([float(v) for v in line.split()[1:]])           # average remaining columns
                       for line in data_lines])
    return x_data, y_data

In [ ]:
# Obtain X and Y data from the JCAMP file
xs_jcamp, ys_jcamp = get_jcamp_data(hcl_jcamp)

# Calculated transmittance spectrum
xs_calc = domain
ys_calc = 1 - spectrum_35  # use H-35Cl only; replace with spectrum_mix if HAS_37CL

# ── Overlay NIST data and calculated spectrum ────────────────────────────
plt.clf()
plt.plot(xs_jcamp, ys_jcamp, label='NIST Experimental', color='black', linewidth=1)
plt.plot(xs_calc,  ys_calc,  label='Calculated (H-35Cl)', color='red',   linewidth=1.5, alpha=0.8)
plt.xlabel('Wavenumber (cm\u207b\u00b9)')
plt.ylabel('Transmittance')
plt.title('HCl Ro-Vibrational Spectrum: NIST vs. Calculated')
plt.legend()
plt.xlim(2500, 3200)
plt.show()

## Questions

### Edit this cell to write your answers below each question.

#### Explain the purpose of prefixes like `np.` and `cons.`

#### Explain the meaning and physical contribution of each term in the \(T(\nu, J)\) equation (refer to the table in the markdown cell above the T_levels definition).

#### How does the Boltzmann population differ between rotational and vibrational energy levels?  Are either well dispersed or isolated?

#### How do we know if we are using enough Boltzmann levels to describe the molecular energy landscape?

#### Describe the selection rules for spectroscopy and the R and P branches.  Which is higher energy?  Why is there a "forbidden transition" (gap) between the two branches?

#### Describe and explain the transition energy scatter plot produced above.

#### This is, principally, a single-molecule analysis.  What effects or interactions that we have not described here can occur to a molecule present in a bulk/ensemble sample?

#### How well does this analysis fit/mimic real spectroscopic data?

#### At what pressure of HCl was the NIST spectrum taken?  What five NIST parameters did you use in the \(T(\nu, J)\) formula?

#### What (in your opinion) was the best assumption made in this analysis?

#### What (in your opinion) was the worst assumption made in this analysis?

## Assignment

#### Choose a Diatomic Molecule and replicate this analysis.
#### No two students may use the same molecule. Post which molecule you will analyze to the discussion on Blackboard.

## Assignment Checklist

Use this checklist to make sure you have completed every required step before submitting.

- [ ] Choose a diatomic molecule (other than HCl) and post your choice to Blackboard
- [ ] Update `mass1` and `mass2` in the constants cell with the correct atomic masses
- [ ] Find the NIST spectroscopic constants for your molecule
- [ ] Fill in `nu_e`, `chi_e`, `B_e`, `D_e`, and `a_e` in the NIST parameters cell
- [ ] Verify the assertion in the NIST parameters cell passes without error
- [ ] Re-run the notebook from top to bottom and confirm all cells execute without error
- [ ] Compare your simulated spectrum to the NIST experimental spectrum and comment on the agreement
- [ ] Answer all reflection questions (cells marked "Your Answer:")
- [ ] Answer all questions in the Questions section at the end
- [ ] (Optional) Fill in H-37Cl constants if your molecule has a stable second isotopologue with measurable abundance